##Cell 1: Setup & Configuration

In [0]:
import logging
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_param(param_name: str, default_value: str) -> str:
    try:
        return dbutils.widgets.get(param_name)
    except Exception:
        return default_value

# Pull from orchestrator (variables.yml) or fallback to local defaults
CATALOG = get_param("catalog_name", "main")
SCHEMA = get_param("schema_name", "fixkart_prd")

# Dynamic Table Paths
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_bookings"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.silver_bookings"
PRIMARY_KEY = "work_order_id"

logger.info(f"Initialized Silver Pipeline | Source: {SOURCE_TABLE} | Target: {TARGET_TABLE}")

## CELL 2: SILVER TRANSFORMATION & DQ LOGIC

In [0]:
def clean_and_deduplicate(spark: SparkSession, source_table: str, primary_key: str) -> DataFrame:
    """
    Reads Bronze data, performs deterministic deduplication, and applies data quality rules.
    """
    logger.info(f"Extracting data from {source_table}...")
    
    try:
        df_bronze = spark.table(source_table)
        
        # 1. Deterministic Deduplication
        logger.info(f"Deduplicating records based on {primary_key}...")
        window_spec = Window.partitionBy(primary_key).orderBy(col("ingested_at").desc())
        
        df_deduped = (
            df_bronze
            .withColumn("rn", row_number().over(window_spec))
            .filter(col("rn") == 1)
            .drop("rn")
        )
        
        # 2. Data Quality Checks
        logger.info("Applying Data Quality (DQ) filters...")
        df_silver = (
            df_deduped
            .filter(col(primary_key).isNotNull())
            .filter(col("technician_id").isNotNull())
        )
        
        logger.info("Silver transformations and DQ rules applied successfully.")
        return df_silver
        
    except Exception as e:
        logger.error(f"Failed during Silver transformation: {str(e)}")
        raise

## CELL 3: DELTA LAKE WRITE UTILITIES (IDEMPOTENT)

In [0]:
def write_silver_layer(spark: SparkSession, df: DataFrame, target_table: str, join_key: str) -> None:
    """
    Writes the cleansed DataFrame to the Silver Delta table using an idempotent MERGE.
    """
    logger.info(f"Initiating write to Silver table '{target_table}'...")
    
    try:
        if spark.catalog.tableExists(target_table):
            logger.info(f"Table {target_table} exists. Performing idempotent MERGE on '{join_key}'...")
            delta_target = DeltaTable.forName(spark, target_table)
            
            (
                delta_target.alias("tgt")
                .merge(
                    df.alias("src"),
                    f"tgt.{join_key} = src.{join_key}"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            logger.info("MERGE operation completed successfully.")
        else:
            logger.info(f"Table {target_table} does not exist. Creating and inserting initial data...")
            (
                df.write.format("delta")
                .option("mergeSchema", "true") 
                .saveAsTable(target_table)
            )
            logger.info("Initial Silver table creation completed successfully.")
            
    except Exception as e:
        logger.error(f"Failed to write to Silver table {target_table}: {str(e)}")
        raise

## CELL 4: ORCHESTRATION & EXECUTION

In [0]:
if 'spark' in locals() or 'spark' in globals():
    try:
        logger.info("--- Starting Silver Processing Pipeline ---")
        
        # 1. Transform & Cleanse
        silver_df = clean_and_deduplicate(
            spark=spark, 
            source_table=SOURCE_TABLE, 
            primary_key=PRIMARY_KEY
        )
        
        # 2. Load (Idempotent)
        write_silver_layer(
            spark=spark,
            df=silver_df, 
            target_table=TARGET_TABLE,
            join_key=PRIMARY_KEY
        )
        
        logger.info("--- Silver Pipeline Completed Successfully ---")
        
    except Exception as main_e:
        logger.error("Pipeline execution aborted due to an error.")
else:
    logger.warning("SparkSession not found. Ensure this code is executing in a Databricks environment.")

## CELL 5: DATA VALIDATION

In [0]:
# Verify the data quality and deduplication
display(spark.table(TARGET_TABLE).limit(20))